# Garbage Classification — EDA
CSE 4830 Group 9

In [ ]:
# ── Step 1: Clone repo ────────────────────────────────────────────────────
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/Garbage-CNN')
if not REPO_DIR.exists():
    os.system('git clone https://github.com/LevinMathew1/Garbage-CNN.git /content/Garbage-CNN')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Working directory:', os.getcwd())

In [ ]:
# ── Step 2: Install missing packages ─────────────────────────────────────
import os
os.system('pip install -q timm')
print('Done.')

In [ ]:
# ── Step 3: Verify GPU ────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Step 4: Mount Google Drive and extract data ───────────────────────────
# Make sure archive.zip is in the root of your Google Drive before running
import zipfile, os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

ZIP_SRC    = Path('/content/drive/MyDrive/archive.zip')
EXTRACT_DIR = Path('/content/data')

if not ZIP_SRC.exists():
    raise FileNotFoundError(
        'archive.zip not found in your Google Drive root.\n'
        'Upload it to drive.google.com first, then re-run this cell.'
    )

if not EXTRACT_DIR.exists():
    print('Extracting...')
    with zipfile.ZipFile(ZIP_SRC) as zf:
        zf.extractall(EXTRACT_DIR)
    print('Done.')
else:
    print('Already extracted.')

print('\nContents of /content/data:')
for p in sorted(EXTRACT_DIR.iterdir()):
    print(' ', p.name)

In [ ]:
# ── Step 5: Resolve class directories ────────────────────────────────────
from pathlib import Path

EXTRACT_DIR = Path('/content/data')
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def find_data_dir(root: Path) -> Path:
    for candidate in [root] + sorted([p for p in root.iterdir() if p.is_dir()]):
        subdirs = sorted([p for p in candidate.iterdir() if p.is_dir()])
        if not subdirs:
            continue
        if set(s.name for s in subdirs) >= {'train'}:
            return candidate
        sample = list(subdirs[0].iterdir())[:10]
        if any(f.suffix.lower() in IMG_EXTS for f in sample):
            return candidate
    raise RuntimeError(f'Cannot find data dir under {root}')

DATA_DIR = find_data_dir(EXTRACT_DIR)
source   = DATA_DIR / 'train' if (DATA_DIR / 'train').is_dir() else DATA_DIR
class_dirs  = sorted([p for p in source.iterdir() if p.is_dir()])
class_names = [p.name for p in class_dirs]

print(f'DATA_DIR = {DATA_DIR}')
print(f'Classes ({len(class_names)}): {class_names}')

In [ ]:
# ── Class distribution bar chart ──────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

all_imgs_per_class = [
    [f for f in d.iterdir() if f.suffix.lower() in IMG_EXTS]
    for d in class_dirs
]
counts = [len(imgs) for imgs in all_imgs_per_class]

print(f'Total images: {sum(counts)}')
for cls, cnt in zip(class_names, counts):
    print(f'  {cls:20s}: {cnt}')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(class_names, counts, color='steelblue')
ax.bar_label(bars)
ax.set_xlabel('Class')
ax.set_ylabel('Image Count')
ax.set_title('Class Distribution — Garbage Classification Dataset')
plt.xticks(rotation=30, ha='right')
fig.tight_layout()

out = Path('outputs/figures')
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / 'class_distribution.png', dpi=150)
plt.show()
print('Saved outputs/figures/class_distribution.png')

In [ ]:
# ── Sample image grid (4 per class) ──────────────────────────────────────
from PIL import Image
import random

random.seed(42)
SAMPLES_PER_CLASS = 4

fig, axes = plt.subplots(
    len(class_names), SAMPLES_PER_CLASS,
    figsize=(SAMPLES_PER_CLASS * 2.5, len(class_names) * 2.5)
)
if len(class_names) == 1:
    axes = [axes]

for row, (imgs, cls_name) in enumerate(zip(all_imgs_per_class, class_names)):
    sample = random.sample(imgs, min(SAMPLES_PER_CLASS, len(imgs)))
    for col in range(SAMPLES_PER_CLASS):
        ax = axes[row][col]
        if col < len(sample):
            ax.imshow(Image.open(sample[col]).convert('RGB'))
        ax.axis('off')
        if col == 0:
            ax.set_title(cls_name, fontsize=9, loc='left')

fig.suptitle('Sample Images per Class', fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# ── Image size distribution ───────────────────────────────────────────────
widths, heights = [], []
for imgs in all_imgs_per_class:
    for p in imgs[:200]:
        try:
            w, h = Image.open(p).size
            widths.append(w)
            heights.append(h)
        except Exception:
            pass

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(widths, bins=30, color='steelblue')
ax1.set_title('Image Width Distribution')
ax1.set_xlabel('Width (px)')
ax2.hist(heights, bins=30, color='salmon')
ax2.set_title('Image Height Distribution')
ax2.set_xlabel('Height (px)')
fig.tight_layout()
plt.show()
print(f'Width  — mean: {np.mean(widths):.0f}px, std: {np.std(widths):.0f}px')
print(f'Height — mean: {np.mean(heights):.0f}px, std: {np.std(heights):.0f}px')

In [ ]:
print('=' * 50)
print('EDA complete.')
print(f'Use DATA_DIR = "{DATA_DIR}" for training.')
print('=' * 50)